## 4. Análise de Dados: Perguntas de Negócio

#### 4.1. Objetivo

Esta é a etapa de análise do pipeline. Até aqui, foi construído e validado toda a infraestrutura de dados (bronze, prata e ouro), já com a qualidade de dados verificada e catálogo documentado. Neste momento então definiremos o motivo disto tudo ter sido realizado: transformando os dados organizados em respostas concretas para as perguntas de negócio que foram definidas no início do projeto.

Cada pergunta será respondida levando em consideração `cubesat_eps.ouro.fato_telemetria_eps`junto com as dimensões `dim_satelite` e `dim_sessao`, ao invés de voltarmos para a camada prata. Esse é o propósito de um modelo em estrela: a tabela fato já traz só as colunas calibradas relevantes para análise e já tem os 8 registros de telemetria de bateria corrompida tratados (nulificados), então as métricas de bateria que calcularmos aqui não serão distorcidas por esse ruído. Juntar fato + dimensões pelas chaves nos dá acesso a `satellite_id`, `constelacao`, `falha_painel_documentada`, `sheet_date` e `is_ground_test` sem precisar repetir lógica de junção em cada pergunta.

In [0]:
# Carrega as tabelas da camada ouro como DataFrames Spark e registra views
# temporárias para permitir tanto SQL quanto PySpark ao longo do notebook.

CATALOG = "cubesat_eps"

df_fato = spark.table(f"{CATALOG}.ouro.fato_telemetria_eps")
df_dim_satelite = spark.table(f"{CATALOG}.ouro.dim_satelite")
df_dim_sessao = spark.table(f"{CATALOG}.ouro.dim_sessao")

df_fato.createOrReplaceTempView("fato_telemetria_eps")
df_dim_satelite.createOrReplaceTempView("dim_satelite")
df_dim_sessao.createOrReplaceTempView("dim_sessao")

print(f"fato_telemetria_eps: {df_fato.count()} linhas")
print(f"dim_satelite: {df_dim_satelite.count()} linhas")
print(f"dim_sessao: {df_dim_sessao.count()} linhas")

fato_telemetria_eps: 32837 linhas
dim_satelite: 4 linhas
dim_sessao: 50 linhas


#### Pergunta 1: Como a tensão/corrente dos paineis solares varia ao longo da órbita e da vida útil do satélite?

Essa pergunta nos entrega duas escalas de tempo distintas (escala orbital - minutos/horas, e escala de vida útil - meses/anos). Sendo assim, precisamos tratá-las de forma separada:

a) Escala orbital (minutos/horas) -> Dentro de uma única sessão de coleta, o satélite passa por ciclos de luz solar direta e eclipse (sombra da terra) a cada órbita. Esperamos ver a tensão e corrente dos paineis subir quando o satélite está iluminado e cair a quase zero na sombra. Usamos `time_stamp_s` (tempo relativo dentro da sessão) para visualizar isso.

b) Escala de vida útil (meses/anos) -> Observamos se a tensão média gerada pelos paineis vai caindo ao longo dos meses/anos em órbita, sinal de degradação por radiação, desgaste térmico, etc.


Antes de visualizar o ciclo orbital, precisamos escolher uma sessão real (não teste em solo) com bastante leituras, para termos resolução suficiente para enxergar o ciclo dia/eclipse.

In [0]:
%sql
SELECT s.sessao_id, s.satellite_id, s.sheet_name, s.sheet_date, COUNT(*) AS n_leituras
FROM fato_telemetria_eps f
JOIN dim_sessao s ON f.sessao_id = s.sessao_id
WHERE s.is_ground_test = false
GROUP BY s.sessao_id, s.satellite_id, s.sheet_name, s.sheet_date
ORDER BY n_leituras DESC
LIMIT 10

sessao_id,satellite_id,sheet_name,sheet_date,n_leituras
50,UGUISU,10 April 2021,2021-04-10,1101
8,TSURU,March 28,2021-03-28,1095
1,NEPALISAT,9 Nov 2020,2020-11-09,1080
49,UGUISU,13 Feb 2021,2021-02-13,1080
3,NEPALISAT,10 April 2021,2021-04-10,1080
5,RAAVANA,11 March 2021,2021-03-11,1080
4,RAAVANA,13 Feb 2021,2021-02-13,1079
48,UGUISU,23 October 2020,2020-10-23,1079
44,TSURU,December 20,2021-12-20,995
2,NEPALISAT,25 Nov 2020,2020-11-25,880


Analisando o resultado acima, optamos por utilizar a sessão de TSURU (Linha número 2) como baseline para o ciclo orbital, uma vez que este satélite não teve falha de painel documentada, então serve como uma boa referência "saudável".

In [0]:
%sql
SELECT
  time_stamp_s,
  vpy_mv, vpx_mv, vpz_mv, vmx_mv, vmz_mv
FROM fato_telemetria_eps
WHERE sessao_id = 8
ORDER BY time_stamp_s

time_stamp_s,vpy_mv,vpx_mv,vpz_mv,vmx_mv,vmz_mv
0.0,1326.31,1272.89,1282.05,1318.68,1288.16
11.3,1326.31,1272.89,1282.05,1318.68,1288.16
22.6,1326.31,1272.89,1282.05,1320.21,1288.16
33.9,1326.31,1272.89,1282.05,1318.68,1286.63
45.2,1326.31,1271.37,1282.05,1318.68,1286.63
56.5,1326.31,1271.37,1280.53,1320.21,1285.1
67.8,1326.31,1269.84,1282.05,1318.68,1285.1
79.1,1326.31,1269.84,1282.05,1318.68,1285.1
90.4,1326.31,1269.84,1282.05,1317.16,1285.1
101.7,1324.79,1269.84,1280.53,1317.16,1286.63


Databricks visualization. Run in Databricks to view.

**Ciclo orbital (dia/eclipse)**

O gráfico gerado na célula acima mostra um padrão semelhante a uma "onda quadrada", os períodos de pico superior ficam próximos de 5000mV (provavelmente os períodos de incidência solar direta) e os valores de base inferior próximos a 1200mV (sombra da terra). Diferentemente de um amanhecer e anoitecer visto da superfície da terra, em órbita a sombra da terra tem uma borda relativamente nítida, o satélite entra e sai do eclipse em poucos segundos, não minutos. Isso explica o motivo dos sinais registrados aparecem com um formato muito semelhante a ondas quadradas.

Podemos notar também que no período de sombra da terra, os satélites registram o valor mediano próximo a 1200mV e isso acontece pois a luz solar ainda pode ser refletida pela terra, além de ruídos no sensor ou uma pequena tensão residual no circuito de condicionamento de potência. 

CubeSats em órbita baixa (aproximadamente 400km da terra) têm período orbital de aproximadamente 95 minutos. Uma janela de 3 horas e 26 minutos cobrindo 2 ciclos completos de luz é compatível com esse período. Além disso, as 5 faces sobem e descem praticamente juntas nessa sessão, o que faz sentido, já que a orientação da CubeSat determina qual face está voltada para o sol em cada momento da órbita (uma análise de qual face domina em cada pico fica para ser respondido na pergunta 5 desta análise).

In [0]:
%sql
SELECT
  s.sheet_date,
  AVG(CASE WHEN s.satellite_id = 'TSURU' THEN (f.vpy_mv+f.vpx_mv+f.vpz_mv+f.vmx_mv+f.vmz_mv)/5 END) AS TSURU,
  AVG(CASE WHEN s.satellite_id = 'RAAVANA' THEN (f.vpy_mv+f.vpx_mv+f.vpz_mv+f.vmx_mv+f.vmz_mv)/5 END) AS RAAVANA,
  AVG(CASE WHEN s.satellite_id = 'UGUISU' THEN (f.vpy_mv+f.vpx_mv+f.vpz_mv+f.vmx_mv+f.vmz_mv)/5 END) AS UGUISU,
  AVG(CASE WHEN s.satellite_id = 'NEPALISAT' THEN (f.vpy_mv+f.vpx_mv+f.vpz_mv+f.vmx_mv+f.vmz_mv)/5 END) AS NEPALISAT
FROM fato_telemetria_eps f
JOIN dim_sessao s ON f.sessao_id = s.sessao_id
WHERE s.is_ground_test = false
GROUP BY s.sheet_date
ORDER BY s.sheet_date

sheet_date,TSURU,RAAVANA,UGUISU,NEPALISAT
2020-10-23,null,null,3048.3397794253783,null
2020-11-09,null,null,null,3388.441896296297
2020-11-25,null,null,null,2940.601961363633
2021-02-13,null,2717.515443929566,3100.85290555555,null
2021-03-11,null,2669.385500000002,null,null
2021-03-28,3120.3793187214696,null,null,null
2021-03-31,3255.787526462396,null,null,null
2021-04-10,null,null,3133.1642016348774,3381.7801648148184
2021-04-20,3684.312843697478,null,null,null
2021-04-27,3672.3320369747908,null,null,null


Databricks visualization. Run in Databricks to view.

O gráfico acima nos mostra para cada satélite, como a tensão média dos paineis (5 faces) mudou de sessão em sessão, ao longo dos meses/anos. No entanto, existe uma limitação importante dos dados: TSURU possui 40 sessões espalhadas por quase um ano inteiro, enquanto que RAAVANA possui apenas 2 sessões, e NEPALISAT e UGUISU possuem 3 sessões. Com 2 ou 3 pontos não é possível afirmar estatisticamente que exista tendência de queda ou subida. 

Tendo isso em mente, olhando o gráfico para os satélites que possuem poucos pontos separadamente, vemos que todos possuem um valor médio próximo da casa de 3100mV, no entanto, conforme mencionado, é impossível afirmarmos alguma tendência com relação a essas informações. Em contrapartida, quando filtramos o gráfico para visualizar apenas o satélite de TSURU conseguimos verificar que ele se mantém numa faixa relativamente estável de operação (3100mV e 3700mV) ao longo de quase um ano inteiro de coleta, sem nenhuma tendência de queda (o que pode coincidir com o fato de este satélite não possuir nenhuma falha documentada).

Em uma das sessões ele apresenta um valor médio de 5100mV, que é um outlier que pode ser interessante de observarmos.

In [0]:
%sql
SELECT
  s.sessao_id, s.sheet_name, s.sheet_date,
  AVG(f.vpy_mv) AS py, AVG(f.vpx_mv) AS px, AVG(f.vpz_mv) AS pz,
  AVG(f.vmx_mv) AS mx, AVG(f.vmz_mv) AS mz,
  AVG((f.vpy_mv+f.vpx_mv+f.vpz_mv+f.vmx_mv+f.vmz_mv)/5) AS media_5_faces
FROM fato_telemetria_eps f
JOIN dim_sessao s ON f.sessao_id = s.sessao_id
WHERE s.satellite_id = 'TSURU' AND s.is_ground_test = false
GROUP BY s.sessao_id, s.sheet_name, s.sheet_date
ORDER BY media_5_faces DESC
LIMIT 5

sessao_id,sheet_name,sheet_date,py,px,pz,mx,mz,media_5_faces
36,October 29,2021-10-29,5112.814860335193,5069.389162011174,5059.720223463685,5105.081061452513,5103.691564245811,5090.139374301674
22,July 17,2021-07-17,4056.036912751681,4379.819697986577,4704.321895973155,3521.5158557046984,3247.2487583892657,3981.788624161073
13,May 13,2021-05-13,3832.864050420164,4029.930067226895,4209.2684873949565,3573.255243697475,3538.4389411764673,3836.7513579831952
34,October 15,2021-10-15,3786.9179495798353,3768.048420168068,3719.1776470588197,3761.422722689078,4052.316084033614,3817.576564705883
43,December 13,2021-12-13,3730.5509174311933,3200.9232721712565,2924.078776758411,4216.907155963306,4797.013455657496,3773.89471559633


O resultado da pesquisa acima nos mostra que todas as 5 faces de painel têm o valor detensão subindo juntos para as 5 sessões com maior tensão média. Isso descarta a hipótesede "sensor com defeito em uma face apenas", se fosse um problema isolado de sensor, esperaríamos ver uma face discrepante das demais, não todas subindo de forma coordenada. Uma hipótese física plausível é que essas sessões tenham ocorrido em períodos de **"beta angle" alto**, uma geometria específica entre a órbita do satélite, a Terra e o Sol em que o satélite permanece em luz solar contínua por dias/semanas, sem entrar na sombra da Terra.

#### Pergunta 2: É possível identificar nos dados, o momento da falha do painel solar em UGUISU e RAAVANA? Quais sinais antecedem a falha?

Conforme mencionado anteriormente, ambos os satélites possuem poucas sessões registradas, não permitindo identificar um momento exato da falha numa linha de tempo contínua. Ao invés de olharmos a média das 5 faces por sessão (que mascararia um problema concentrado numa única fase) vamos olhar para o valor máximo de tensão em cada face individual, dentro de cada sessão. Em uma face saudável, em algum momento da sessão ela deve atingir a tensão de pico de iluminação plena (perto de 5000mV). Se uma face está com um painel danificado seu máximo dentro de uma sessão ficaria bem abaixo disto, mesmo durante os períodos de luz solar.

In [0]:
%sql
SELECT
  s.satellite_id, s.sheet_date, s.sheet_name,
  MAX(f.vpy_mv) AS max_py, MAX(f.vpx_mv) AS max_px, MAX(f.vpz_mv) AS max_pz,
  MAX(f.vmx_mv) AS max_mx, MAX(f.vmz_mv) AS max_mz
FROM fato_telemetria_eps f
JOIN dim_sessao s ON f.sessao_id = s.sessao_id
WHERE s.satellite_id IN ('UGUISU', 'RAAVANA') AND s.is_ground_test = false
GROUP BY s.satellite_id, s.sheet_date, s.sheet_name
ORDER BY s.satellite_id, s.sheet_date

satellite_id,sheet_date,sheet_name,max_py,max_px,max_pz,max_mx,max_mz
RAAVANA,2021-02-13,13 Feb 2021,4.58,5503.66,5477.72,5422.77,5467.03
RAAVANA,2021-03-11,11 March 2021,4.58,5402.93,5325.09,5331.2,5381.56
UGUISU,2020-10-23,23 October 2020,3110.5,5401.4,5405.98,5409.04,5361.72
UGUISU,2021-02-13,13 Feb 2021,3064.71,5419.72,5453.3,5470.09,5453.3
UGUISU,2021-04-10,10 April 2021,3034.19,5402.93,5392.25,5428.88,5430.4


A análise anterior apontou que a face +Y da UGUISU nunca passa de +3110.5mV, enquanto que as outras faces chegam ao pico. Um número agregago, porém, esconde como o problema se manifesta ao longo da órbita. Por isso, voltamos ao dado bruto da série temporal: as cinco tensões de painel ao longo de uma sessão inteira.

In [0]:
%sql
SELECT
  f.time_stamp_s / 60 AS tempo_min,
  f.vpy_mv AS `+Y`,
  f.vpx_mv AS `+X`,
  f.vmx_mv AS `-X`,
  f.vpz_mv AS `+Z`,
  f.vmz_mv AS `-Z`
FROM fato_telemetria_eps f
JOIN dim_sessao s ON f.sessao_id = s.sessao_id
WHERE s.satellite_id = 'UGUISU'
  AND s.sheet_date = (SELECT MAX(sheet_date) FROM dim_sessao WHERE satellite_id = 'UGUISU')
ORDER BY f.time_stamp_s

tempo_min,+Y,+X,-X,+Z,-Z
0.0,2414.53,5152.63,5097.68,5103.79,5106.84
0.08333333333333333,2405.37,5137.36,5079.37,5083.94,5097.68
0.16666666666666666,2397.74,5161.78,5099.21,5103.79,5106.84
0.25,2388.58,5189.26,5123.63,5116.0,5116.0
0.3333333333333333,2377.9,5224.36,5146.52,5140.42,5122.1
0.4166666666666667,2368.74,5253.36,5166.36,5149.57,5122.1
0.5,2358.06,5277.78,5178.57,5152.63,5122.1
0.5833333333333334,2345.85,5305.25,5184.68,5157.2,5120.57
0.6666666666666666,2335.16,5311.36,5190.78,5158.73,5117.52
0.75,2321.43,5318.99,5190.78,5160.26,5117.52


Databricks visualization. Run in Databricks to view.

É possível identificar o momento de falha? Somente em parte. Os dados mostram com clareza qual painel falhou nos dois satélites, mas não o momento exato da falha. Em RAAVANA a face +Y está totalmente inoperante (tensão máxima de 4.6mV), já na primeira sessão disponível. A falha aconteceu antes da janela de dados, então não dá para localizar o momento nem observar sinais precursores.

Em UGUISU, a face +Y apresenta degradação parcial e progressiva. A tensão máxima cai de 3110 -> 3065 -> 3034mV continuamente nas medidas, enquanto as outras fases atingem 5400mV. O gráfico do ciclo orbital mostra que a face continua acompanhando o ciclo sol/eclipse: o piso no eclipse é igual as outras faces, o que indica que o sensor funciona, mas o teto na fase iluminada fica em torno da metade. É o comportamento esperado de um painel com parte das células danificadas, não de uma falha total. Revisado na Pergunta 5: a análise de corrente mostrou que a +Y da UGUISU entrega no máximo 65 mA (contra ~500 mA nas faces saudáveis), ou seja, a face está funcionalmente quase inoperante, e não apenas parcialmente degradada.

**Quais sinais antecedem a falha?**

Os dados sugerem que a tensão sozinha pode mascarar a gravidade do problema: a UGUISU ainda mostra tensão razoável na +Y, mas praticamente não entrega corrente. O indicador mais confiável para a operação em solo é a potência (V × I) por face e por sessão, acompanhada ao longo do tempo. Uma queda sustentada da potência de uma face específica, com as demais normais, é o sinal a monitorar.

### Pergunta 3: Comparação do desempenho do EPS entre os quatro satélites

Os quatro satélites da constelação BIRDS utilizam o mesmo projeto de CubeSat, então em condições normais deveriam apresentar desempenho parecido. Diferenças relevantes entre eles indicam problemas de hardware (como as falhas de painel já mencionadas) ou condições de operação distintas. Para podermos comparar-las definimos "desempenho de EPS" em duas formas:

- Geração de energia (paineis solares): calculamos a potência instantânea total gerada pelas cinco faces:

$$P_{total} = \sum_{faces} \frac{V_{face}\,[\text{mV}] \times I_{face}\,[\text{mA}]}{1000} \quad [\text{mW}]$$

A potência é uma métrica melhor que a tensão sozinha porque apresenta a energia efetivamente entregue ao satélite: um painel pode ter tensão alta e fornecer corrente baixa (ou vice-versa). Reportamos a potência média e a máxima.

- Armazenamento (bateria): tensão média e mínima (quanto a bateria descarrega durante os eclipses, que é quando ela sustenta o satélite sozinha) e a faixa de temperatura.

**Observações**

- Foram excluidas as sessões de teste em solo.
- As leituras de bateria corrompidas já estão como NULL.
- O número de sessões varia muito entre satélites.


In [0]:
%sql
SELECT
  s.satellite_id,
  COUNT(DISTINCT s.sessao_id) AS sessoes,
  COUNT(*) AS leituras,
  ROUND(AVG((f.vpy_mv*f.ipy_ma + f.vpx_mv*f.ipx_ma + f.vmx_mv*f.imx_ma
           + f.vpz_mv*f.ipz_ma + f.vmz_mv*f.imz_ma) / 1000), 0) AS potencia_media_mw,
  ROUND(MAX((f.vpy_mv*f.ipy_ma + f.vpx_mv*f.ipx_ma + f.vmx_mv*f.imx_ma
           + f.vpz_mv*f.ipz_ma + f.vmz_mv*f.imz_ma) / 1000), 0) AS potencia_max_mw,
  ROUND(AVG(f.vbat_v), 3) AS vbat_media_v,
  ROUND(MIN(f.vbat_v), 3) AS vbat_min_v,
  ROUND(MIN(f.tbat_c), 1) AS tbat_min_c,
  ROUND(MAX(f.tbat_c), 1) AS tbat_max_c
FROM fato_telemetria_eps f
JOIN dim_sessao s ON f.sessao_id = s.sessao_id
WHERE s.is_ground_test = false
GROUP BY s.satellite_id
ORDER BY s.satellite_id

satellite_id,sessoes,leituras,potencia_media_mw,potencia_max_mw,vbat_media_v,vbat_min_v,tbat_min_c,tbat_max_c
NEPALISAT,3,3040,1055.0,4184.0,4.088,3.87,1.1,8.7
RAAVANA,2,2159,1085.0,3772.0,4.086,3.87,2.2,9.1
TSURU,40,23266,1079.0,4378.0,4.121,3.79,-0.1,24.4
UGUISU,3,3260,1022.0,3944.0,4.095,3.89,-22.7,6.8


Databricks visualization. Run in Databricks to view.

A potência média é praticamente a mesma nos quatro satélites, o que é contraintuitivo, uma vez que em RAAVANA há uma face morta, e ela possui ainda a maior média. Podemos pensar nisso como o fato de o satélite girar, e em cada momento só 1 a 3 faces estão voltadas para o sol. Perder uma face somente causaria prejuízos quando é a vez de ela estar iluminada, e na média de uma órbita inteira isso se dilui. Além disso, a geração depende da demanda, quando a bateria está cheia, o EPS não puxa corrente máxima dos paineis. A média reflete mais o quanto o satélite precisou de energia do que o quanto ele conseguiria gerar.

Ao olharmos a coluna de **potência máxima** vemos que os dois satélites que possuem falha documentada apresentam o menor pico, e a RAAVANA (face morta) fica abaixo da UGUISU (face degradada). A ressalva fica pelo fato da TSURU possuir 40 sessões, e consequentemente, possuir muito mais "chances" de atingir um pico alto.

Um ponto de suspeita fica por conta da `tbat_min = -22,7 °C` em UGUISU: antes de interpretarmos isso, precisamos saber se é uma leitura isolada ou uma queda real de temperatura durante vários minutos.

In [0]:
%sql
SELECT s.sheet_date, s.sheet_name,
       COUNT(*) AS leituras_abaixo_zero,
       MIN(f.tbat_c) AS tbat_min,
       MIN(f.vbat_v) AS vbat_min,
       MIN(f.time_stamp_s) AS primeiro_s,
       MAX(f.time_stamp_s) AS ultimo_s
FROM fato_telemetria_eps f
JOIN dim_sessao s ON f.sessao_id = s.sessao_id
WHERE s.satellite_id = 'UGUISU' AND s.is_ground_test = false
  AND f.tbat_c < 0
GROUP BY s.sheet_date, s.sheet_name
ORDER BY s.sheet_date

sheet_date,sheet_name,leituras_abaixo_zero,tbat_min,vbat_min,primeiro_s,ultimo_s
2020-10-23,23 October 2020,1,-22.68,4.2,0.0,0.0
2021-04-10,10 April 2021,15,-0.08,4.2,5400.0,5485.0


**Conclusão**

A potência média é praticamente igual nos quatro satélites, inclusive na RAAVANA, que possui uma face inoperante. Isso possui duas explicações principais:

1. O satélite gira, e a cada instante só algumas faces estão iluminadas, então a perda de uma face se dilui na média da órbita. 

2. A geração é limitada pela demanda, com a bateria carregada, o EPS não extrai a potência máxima dos paineis, por isso a média não é um bom indicador de falha.

Já a potência máxima representa a capacidade de pico, separando claramente os satélites:

| Satélite | Potência máx. (mW) | Diferença vs TSURU |
|---|---|---|
| TSURU | 4.378 | — |
| NEPALISAT | 4.184 | −4% |
| UGUISU | 3.944 | −10% |
| RAAVANA | 3.772 | −14% |

Os dois satélites com falha documentada têm os menores picos, e a ordem é coerente com a gravidade encontrada em P2: RAAVANA (com face +Y inoperante) fica abaixo da UGUISU (face +Y degradada), com a ressalva novamente de que TSURU possui 40 sessões em órbita contra 2 a 3 das demais, o que aumenta a chance de ela registrar picos altos.

A bateria é saudável nos quatro satélites. A tensão média fica entre 4,09 e 4,12 V e a mínima nunca fica abaixo de 3,79 V. Ou seja, mesmo com paineis comprometidos, o balanço de energia se manteve positivo no período observado: a falha reduziu a margem de geração, mas não comprometeu a operação.

A temperatura da bateria apresenta um valor mínimo de -22,7 °C em UGUISU, o que consideramos como uma leitura espúria, pois possui um único registro no instante 0 da sessão de 23/10/2020, isolado entre leituras acima de 0 °C. Trata-se provavelmente de um frame corrompido que não foi capturado pelo critério de limpeza da etapa de qualidade, e por isso deve ser desconsiderado. Desprezando essa leitura, as temperaturas ficam entre 0 e 24 °C, faixa adequada.

### Pergunta 4: Degradação da bateria ao longo do tempo

Baterias do tipo íon-lítio envelhecem em ciclos de carga e descarga e com a temperatura. Em órbita baixa, um CubeSat passa por um ciclo completo a cada órbita (aproximadamente 100 minutos): carrega no sol e descarrega no eclipse, o que dá cerca de 15 ciclos por dia. Em poucos meses, a bateria acumula milhares de ciclos.

A principal consequência do envelhecimento é a perda de capacidade, ou seja, a bateria armazena menos carga. Para a mesma demanda de energia no eclipse, uma bateria degradada descarrega mais profundamente, então a sua tensão mínima fica mais baixa. Também pode haver queda de tensão máxima, se a bateria deixa de atingir a carga plena. Por isso, acompanhamos por sessão:

- tensão mínima.
- tensão média e máxima.
- temperatura média.

**Observação**: Usaremos apenas a TSURU, único satélite que possui cobertura temporal suficiente (40 sessões em órbitas ao longo de 10 meses).

In [0]:
%sql
SELECT
  s.sheet_date,
  ROUND(MIN(f.vbat_v), 3) AS vbat_min_v,
  ROUND(AVG(f.vbat_v), 3) AS vbat_media_v,
  ROUND(MAX(f.vbat_v), 3) AS vbat_max_v,
  ROUND(AVG(f.tbat_c), 1) AS tbat_media_c
FROM fato_telemetria_eps f
JOIN dim_sessao s ON f.sessao_id = s.sessao_id
WHERE s.satellite_id = 'TSURU' AND s.is_ground_test = false
GROUP BY s.sheet_date
ORDER BY s.sheet_date

sheet_date,vbat_min_v,vbat_media_v,vbat_max_v,tbat_media_c
2021-03-28,3.92,4.109,4.2,8.3
2021-03-31,3.92,4.081,4.2,7.3
2021-04-20,3.94,4.123,4.2,2.8
2021-04-27,3.94,4.12,4.2,2.4
2021-05-06,3.97,4.127,4.2,3.0
2021-05-13,4.1,4.176,4.2,13.8
2021-05-21,3.94,4.152,4.2,8.1
2021-05-28,3.94,4.115,4.2,3.0
2021-06-01,3.89,4.079,4.2,3.0
2021-06-11,3.94,4.119,4.2,2.0


Databricks visualization. Run in Databricks to view.

Não há evidência de degradação progressiva da bateria da TSURU no período observado (10 meses e cerca de 4500 ciclos orbitais estimados). A tensão máxima permanece a mesma em todas as 40 sessões observadas, e a bateria atinge carga plena do início ao fim do período. A tensão mínima permanece estável dentro de uma faixa comum de operação, sem tendência de queda ao longo dos meses. Se a bateria estivesse perdendo capacidade, esperaríamos descargas cada vez mais profundas no eclipse, ou seja, mínimas cada vez menores.

**Limitações**

- A tensão mínima é um indicador indireto de capacidade. Uma medida direta exigiria integrar a corrente ao longo de descargas completas.

- Apenas a TSURU tem cobertura suficiente. Nos demais satélites (2 a 3 sessões) não é possível avaliar tendência.

### Pergunta 5: Contribuição de cada face para a geração de energia

Um CubeSat 1U tem painei solares em várias faces (nos satélites BIRDS +X, +Y, +Z, -X e -Z). Como o satélite gira e muda a orientação ao longo da órbita, cada face recebe uma quantidade diferente de luz solar. Face que contribui mais depende principalmente do controle de atitude (como o satélite está orientado em relação ao sol) e da saúde de cada painel.

Para isso, calculamos a potência de cada face em cada leitura, por satélite, e reportamos:

- Potência média por face.

- Participação percentual da energia que veio daquela face.

Expectativa: nos satélites com falha da face +Y, a sua participação deve ser inferior. A TSURU, com mais dados e sem falha documentada, é a melhor referência de comportamento normal.

In [0]:
%sql
WITH potencia_por_face AS (
  SELECT
    s.satellite_id,
    stack(5,
      '+Y', f.vpy_mv * f.ipy_ma / 1000,
      '+X', f.vpx_mv * f.ipx_ma / 1000,
      '-X', f.vmx_mv * f.imx_ma / 1000,
      '+Z', f.vpz_mv * f.ipz_ma / 1000,
      '-Z', f.vmz_mv * f.imz_ma / 1000
    ) AS (face, potencia_mw)
  FROM fato_telemetria_eps f
  JOIN dim_sessao s ON f.sessao_id = s.sessao_id
  WHERE s.is_ground_test = false
)
SELECT
  satellite_id,
  face,
  ROUND(AVG(potencia_mw), 0) AS potencia_media_mw,
  ROUND(100 * SUM(potencia_mw) / SUM(SUM(potencia_mw)) OVER (PARTITION BY satellite_id), 1) AS participacao_pct
FROM potencia_por_face
GROUP BY satellite_id, face
ORDER BY satellite_id, participacao_pct DESC

satellite_id,face,potencia_media_mw,participacao_pct
NEPALISAT,+X,256.0,24.3
NEPALISAT,+Y,243.0,23.0
NEPALISAT,-X,242.0,22.9
NEPALISAT,-Z,221.0,20.9
NEPALISAT,+Z,94.0,8.9
RAAVANA,-Z,298.0,27.5
RAAVANA,-X,292.0,26.9
RAAVANA,+X,263.0,24.3
RAAVANA,+Z,231.0,21.3
RAAVANA,+Y,0.0,0.0


Databricks visualization. Run in Databricks to view.

Com base no resultado anterior podemos ver que nenhuma face domina sozinha. Nas faces saudáveis a participação fica entre 20 a 30%. Isso combina com um satélite que gira (sem apontamento fixo para o sol): a luz vai se distribuindo entre as faces. As faces X são as mais consistentes, aparecendo sempre entre as 3 primeiras nos quatro satélites, enquanto que a +Z é sempre a mais fraca (uma hipótese seria dizer que esta face possui menos área de painel, por dividir espaço com antena, câmera, ou outro componente).

A surpresa fica para UGUISU apresentando um valor de 9mW (0,9%). Na P2 vimos que a tensão desta face neste satélite chega a metade das outras e cicla normalmente, mas a sua potência é praticamente zero (acusando um valor de corrente muito próximo de zero). Este é um comportamento típico de um painel com circuito aberto (conexão rompida): a célula ainda produz tensão quando iluminada, mas não entrega corrente para o sistema. Aí a face está funcionalmente morta também, e não parcialmente degradada como escrevemos na P2.

In [0]:
%sql
SELECT s.satellite_id,
       ROUND(MAX(f.ipy_ma), 1) AS max_ipy_ma,
       ROUND(AVG(f.ipy_ma), 1) AS media_ipy_ma,
       ROUND(MAX(f.ipx_ma), 1) AS max_ipx_ma,
       ROUND(AVG(f.ipx_ma), 1) AS media_ipx_ma
FROM fato_telemetria_eps f
JOIN dim_sessao s ON f.sessao_id = s.sessao_id
WHERE s.is_ground_test = false
GROUP BY s.satellite_id
ORDER BY s.satellite_id

satellite_id,max_ipy_ma,media_ipy_ma,max_ipx_ma,media_ipx_ma
NEPALISAT,522.0,48.2,536.0,51.2
RAAVANA,51.3,19.5,512.7,53.8
TSURU,540.7,57.0,503.4,42.1
UGUISU,65.3,4.7,508.1,67.8


**Conclusão**

A análise da corrente confirmou que a +Y da UGUISU, apesar de ainda apresentar uma tensão média de 3100mV, entrega no máximo 65mA (contra 500mA nas outras faces saudáveis). Este resultado revisou a conclusão da pergunta 2: a face está funcionalmente quase inoperante, e não apenas parcialmente degradada.